In [1]:
import pandas as pd
from sklearn.model_selection import(
    train_test_split,
    GridSearchCV
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib

In [2]:
df = pd.read_csv("../data/preprocessed_housing.csv")

In [3]:
X = df.drop(columns=["Id", "SalePrice"])
y = df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

TUNING DECISION TREE

In [4]:
dt_params = {
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
dt_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=dt_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)
print("Best Parameters:", dt_grid.best_params_)
print("Best Cross-Validation R²:", dt_grid.best_score_)

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best Cross-Validation R²: 0.7140115253275854


In [5]:
best_dt = dt_grid.best_estimator_

y_pred_dt = best_dt.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = mse_dt ** 0.5
r2_dt = r2_score(y_test, y_pred_dt)

print(f"MAE : {mae_dt:.2f}")
print(f"MSE : {mse_dt:.2f}")
print(f"RMSE: {rmse_dt:.2f}")
print(f"R²  : {r2_dt:.4f}")

MAE : 27256.44
MSE : 1691134831.90
RMSE: 41123.41
R²  : 0.7795


TUNING RANDOM FOREST

In [6]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print("Best Parameters:", rf_grid.best_params_)
print("Best Cross-Validation R²:", rf_grid.best_score_)

Best Parameters: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best Cross-Validation R²: 0.8406165476515934


In [7]:
best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mse_rf ** 0.5
r2_rf = r2_score(y_test, y_pred_rf)

print(f"MAE : {mae_rf:.2f}")
print(f"MSE : {mse_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²  : {r2_rf:.4f}")

MAE : 17711.51
MSE : 842210628.25
RMSE: 29020.87
R²  : 0.8902


TUNING XGBOOST

In [8]:
xgb_params = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}
xgb_grid = GridSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),
    param_grid=xgb_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
print("Best Parameters:", xgb_grid.best_params_)
print("Best Cross-Validation R²:", xgb_grid.best_score_)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best Cross-Validation R²: 0.8622119069099426


In [9]:
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = mse_xgb ** 0.5
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"MAE : {mae_xgb:.2f}")
print(f"MSE : {mse_xgb:.2f}")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"R²  : {r2_xgb:.4f}")

MAE : 16183.15
MSE : 631471808.00
RMSE: 25129.10
R²  : 0.9177


SAVING THE BEST MODEL

In [10]:
joblib.dump(best_xgb,"../models/best_xgboost_model.pkl")

['../models/best_xgboost_model.pkl']

In [11]:
joblib.dump(X.columns.tolist(), "../models/feature_columns.pkl")

['../models/feature_columns.pkl']

In [12]:
model_info = {
    "model": "XGBoost Regressor",
    "r2_score": 0.9177,
    "mae": 16183.15,
    "rmse": 25129.10,
    "best_params": xgb_grid.best_params_
}

joblib.dump(model_info, "../models/model_info.pkl")

['../models/model_info.pkl']